# Phase 2 - Notebook 05: 3DGS+SLAM Methods Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/05_gs_slam_comparison.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the landscape of 3DGS+SLAM methods
2. Compare SplaTAM, GS-SLAM, MonoGS, and Photo-SLAM
3. Know the input requirements for each method
4. Understand different tracking mechanisms
5. Compare mapping strategies and Gaussian update approaches
6. Evaluate performance on standard benchmarks
7. Know when to use which method

**Estimated Time**: 90 minutes

**Prerequisites**: Notebooks 01-04 (SLAM basics, SplaTAM architecture, map initialization, Gaussian update)

---

## 0. Environment Setup

In [ ]:
# Environment setup
import os
import sys

# Colab compatibility
if 'COLAB_GPU' in os.environ:
    !pip install -q matplotlib numpy pandas seaborn
    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    %cd 3DGS-from-scratch

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("Environment ready!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

## 1. 3DGS+SLAM Methods Overview

2023-2024 saw rapid development of 3DGS+SLAM methods. Let's understand their evolution.

In [ ]:
# Timeline visualization
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')

# Timeline
ax.plot([1, 15], [6, 6], 'k-', linewidth=3)
ax.text(8, 6.8, '2023-2024: 3DGS+SLAM Methods Timeline', ha='center', fontsize=16, fontweight='bold')

# Methods data
methods_timeline = [
    {'name': 'GS-SLAM', 'date': 'Nov 2023', 'venue': 'CVPR 2024', 'x': 3, 'y': 8.5,
     'color': '#2E7D32', 'features': ['Coarse-to-fine', 'Adaptive Strategy', 'Faster']},
    {'name': 'SplaTAM', 'date': 'Dec 2023', 'venue': 'CVPR 2024', 'x': 6, 'y': 8.5,
     'color': '#1565C0', 'features': ['First 3DGS SLAM', 'RGB-D Input', 'Dense Tracking']},
    {'name': 'MonoGS', 'date': 'Dec 2023', 'venue': 'CVPR 2024', 'x': 9, 'y': 8.5,
     'color': '#E65100', 'features': ['Monocular Input', 'Depth Estimation', 'No RGB-D']},
    {'name': 'Photo-SLAM', 'date': 'Feb 2024', 'venue': 'CVPR 2024', 'x': 12, 'y': 8.5,
     'color': '#6A1B9A', 'features': ['Hybrid Approach', 'Feature Tracking', 'Superpoints']},
]

for method in methods_timeline:
    # Connection line
    ax.plot([method['x'], method['x']], [6, method['y']-0.8], 'k--', linewidth=1.5, alpha=0.5)
    ax.plot(method['x'], 6, 'ko', markersize=10)
    
    # Method box
    box = FancyBboxPatch((method['x']-1.2, method['y']-0.8), 2.4, 2.5,
                         boxstyle="round,pad=0.1",
                         facecolor='white', edgecolor=method['color'], linewidth=3)
    ax.add_patch(box)
    
    # Method name
    ax.text(method['x'], method['y']+1.2, method['name'], ha='center', fontsize=13, fontweight='bold',
            color=method['color'])
    
    # Date and venue
    ax.text(method['x'], method['y']+0.9, method['date'], ha='center', fontsize=9, style='italic')
    ax.text(method['x'], method['y']+0.6, method['venue'], ha='center', fontsize=8, color='gray')
    
    # Features
    for i, feat in enumerate(method['features']):
        ax.text(method['x'], method['y']+0.2-i*0.25, f'• {feat}', ha='center', fontsize=8)

# Bottom: Core question
problem_box = FancyBboxPatch((1, 0.5), 14, 2.2, boxstyle="round,pad=0.1",
                             facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(problem_box)
ax.text(8, 2.3, 'Core Question: How to replace traditional SLAM map representation with 3DGS?', ha='center', fontsize=12, fontweight='bold')
ax.text(8, 1.8, 'Traditional SLAM: Sparse/Dense Point Cloud  ->  3DGS+SLAM: Explicit Gaussian Ellipsoids', ha='center', fontsize=11)
ax.text(8, 1.3, 'Advantages: Photorealistic rendering + Real-time performance + Editability', ha='center', fontsize=11, color='#E65100')

plt.tight_layout()
plt.savefig('gs_slam_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n3DGS+SLAM Methods Timeline:")
print("  • GS-SLAM: Optimized tracking and mapping strategies (Nov 2023)")
print("  • SplaTAM: First complete 3DGS SLAM system (Dec 2023)")
print("  • MonoGS: First monocular 3DGS SLAM (Dec 2023)")
print("  • Photo-SLAM: Hybrid feature tracking + neural rendering (Feb 2024)")

## 2. Comprehensive Comparison Table

Let's systematically compare the four methods across all dimensions.

In [ ]:
# Create comparison table
comparison_data = {
    'Dimension': [
        'Publication',
        'Conference',
        'Input Type',
        'Tracking Method',
        'Mapping Strategy',
        'Rendering',
        'Real-time',
        'Loop Closure',
        'Memory Usage',
        'Open Source',
    ],
    'SplaTAM': [
        'Dec 2023',
        'CVPR 2024',
        'RGB-D',
        'Differentiable Rendering',
        'Online Gaussian Update',
        'Rasterization',
        '10 FPS tracking',
        'None (implicit)',
        'Medium',
        'Yes',
    ],
    'GS-SLAM': [
        'Nov 2023',
        'CVPR 2024',
        'RGB-D',
        'Diff Render + Coarse-to-fine',
        'Adaptive Expansion',
        'Rasterization',
        '20 FPS tracking',
        'None',
        'Medium',
        'No',
    ],
    'MonoGS': [
        'Dec 2023',
        'CVPR 2024',
        'RGB Monocular',
        'Differentiable Rendering',
        'Monocular Depth Estimation',
        'Rasterization',
        '15 FPS tracking',
        'Yes',
        'High',
        'Yes',
    ],
    'Photo-SLAM': [
        'Feb 2024',
        'CVPR 2024',
        'RGB',
        'Feature Point Tracking',
        'Keyframe Gaussians',
        'Hybrid Rendering',
        'Real-time',
        'Yes',
        'Medium',
        'Yes',
    ],
}

# Display table
df = pd.DataFrame(comparison_data)
df = df.set_index('Dimension')

print("="*100)
print("3DGS+SLAM Methods Comparison Table".center(100))
print("="*100)
print(df.T.to_string())
print("="*100)

## 3. SplaTAM Deep Dive

SplaTAM is the first complete 3DGS+SLAM system, establishing the foundation for this field.

In [ ]:
# SplaTAM Architecture
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('SplaTAM: System Architecture', fontsize=16, fontweight='bold', pad=20)

# Input
input_box = FancyBboxPatch((5.5, 10), 3, 1.2, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(7, 10.6, 'RGB-D Input', ha='center', fontsize=12, fontweight='bold')

# Tracking Module
track_box = FancyBboxPatch((1, 7), 5, 2.5, boxstyle="round,pad=0.15",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)
ax.add_patch(track_box)
ax.text(3.5, 9, 'Tracking Module', ha='center', fontsize=13, fontweight='bold', color='#E65100')

track_steps = [
    '1. Render from pose estimate',
    '2. Compare with observation',
    '3. Optimize pose (gradient descent)',
]
for i, step in enumerate(track_steps):
    ax.text(3.5, 8.4-i*0.35, step, ha='center', fontsize=9)

# Mapping Module
map_box = FancyBboxPatch((8, 7), 5, 2.5, boxstyle="round,pad=0.15",
                         facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=3)
ax.add_patch(map_box)
ax.text(10.5, 9, 'Mapping Module', ha='center', fontsize=13, fontweight='bold', color='#2E7D32')

map_steps = [
    '1. Add new Gaussians from depth',
    '2. Optimize Gaussians (render loss)',
    '3. Silhouette-guided densification',
]
for i, step in enumerate(map_steps):
    ax.text(10.5, 8.4-i*0.35, step, ha='center', fontsize=9)

# Gaussian Map
map_data_box = FancyBboxPatch((5, 4), 4, 2, boxstyle="round,pad=0.1",
                              facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=2)
ax.add_patch(map_data_box)
ax.text(7, 5.5, 'Gaussian Map', ha='center', fontsize=12, fontweight='bold', color='#6A1B9A')
ax.text(7, 5.0, '{mu, S, R, sigma, SH}', ha='center', fontsize=10, style='italic')
ax.text(7, 4.5, 'Explicit Gaussian Representation', ha='center', fontsize=9)

# Arrows
ax.annotate('', xy=(3.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(10.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(6, 6), xytext=(3.5, 7),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
ax.text(4.5, 6.5, 'Pose', fontsize=9, color='#E65100')
ax.annotate('', xy=(8, 6), xytext=(10.5, 7),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.text(9.5, 6.5, 'Update', fontsize=9, color='#2E7D32')
ax.annotate('', xy=(3.5, 7), xytext=(5, 4),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=1.5, linestyle='--'))

# Output
output_box = FancyBboxPatch((5.5, 1.5), 3, 1.2, boxstyle="round,pad=0.1",
                            facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2)
ax.add_patch(output_box)
ax.text(7, 2.4, 'Outputs', ha='center', fontsize=11, fontweight='bold')
ax.text(7, 1.9, 'Camera Trajectory + Dense Map', ha='center', fontsize=9)

ax.annotate('', xy=(7, 2.7), xytext=(7, 4),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Innovation
innovation_box = FancyBboxPatch((0.3, 0.3), 13.4, 1, boxstyle="round,pad=0.1",
                                facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(innovation_box)
ax.text(7, 0.95, 'Key Innovation: Differentiable Rendering for Tracking', ha='center', fontsize=11, fontweight='bold')
ax.text(7, 0.55, 'Replace geometric error with rendering error for end-to-end differentiable tracking', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('splatam_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSplaTAM Core Features:")
print("  1. First complete 3DGS+SLAM system")
print("  2. Differentiable rendering tracking: minimize rendering error to optimize camera pose")
print("  3. Silhouette-guided densification: add Gaussians based on rendering coverage")
print("  4. Online Gaussian update: add and optimize Gaussian parameters in real-time")

### 3.1 SplaTAM Tracking Mechanism

SplaTAM's tracking is based on differentiable rendering - the biggest difference from traditional SLAM.

In [ ]:
# Tracking comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Traditional SLAM
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Traditional SLAM Tracking\n(ORB-SLAM2)', fontsize=12, fontweight='bold')

steps_trad = ['Feature Extraction', 'Feature Matching', 'PnP/RANSAC', 'Pose Refinement']
y_pos = 8
for i, step in enumerate(steps_trad):
    box = FancyBboxPatch((2, y_pos-i*1.5), 6, 1, boxstyle="round,pad=0.1",
                         facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
    ax.add_patch(box)
    ax.text(5, y_pos-i*1.5+0.5, step, ha='center', va='center', fontsize=10)
    if i < len(steps_trad)-1:
        ax.annotate('', xy=(5, y_pos-(i+1)*1.5+0.9), xytext=(5, y_pos-i*1.5-0.1),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(5, 1, 'Sparse Features +\nGeometric Error', ha='center', fontsize=9, style='italic', color='#1565C0')

# Direct SLAM
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Direct SLAM Tracking\n(DSO)', fontsize=12, fontweight='bold')

steps_direct = ['Image Alignment', 'Photometric Error', 'Gauss-Newton', 'Pose Update']
y_pos = 8
for i, step in enumerate(steps_direct):
    box = FancyBboxPatch((2, y_pos-i*1.5), 6, 1, boxstyle="round,pad=0.1",
                         facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
    ax.add_patch(box)
    ax.text(5, y_pos-i*1.5+0.5, step, ha='center', va='center', fontsize=10)
    if i < len(steps_direct)-1:
        ax.annotate('', xy=(5, y_pos-(i+1)*1.5+0.9), xytext=(5, y_pos-i*1.5-0.1),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(5, 1, 'Photometric Error +\nPoint Cloud Map', ha='center', fontsize=9, style='italic', color='#E65100')

# SplaTAM
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('SplaTAM Tracking\n(3DGS-based)', fontsize=12, fontweight='bold')

steps_splat = ['Render Gaussians', 'Rendering Error', 'Backpropagation', 'Pose Gradient']
y_pos = 8
for i, step in enumerate(steps_splat):
    box = FancyBboxPatch((2, y_pos-i*1.5), 6, 1, boxstyle="round,pad=0.1",
                         facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
    ax.add_patch(box)
    ax.text(5, y_pos-i*1.5+0.5, step, ha='center', va='center', fontsize=10)
    if i < len(steps_splat)-1:
        ax.annotate('', xy=(5, y_pos-(i+1)*1.5+0.9), xytext=(5, y_pos-i*1.5-0.1),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(5, 1, 'Differentiable Rendering +\nGaussian Map', ha='center', fontsize=9, style='italic', color='#2E7D32')

plt.tight_layout()
plt.savefig('tracking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTracking Mechanism Comparison:")
print("  Traditional SLAM: Feature matching -> PnP -> Geometric error")
print("  Direct Method: Photometric error -> Point cloud map")
print("  SplaTAM: Rendering error -> Gaussian map (differentiable)")
print("\nSplaTAM Advantage: Rendering is differentiable, enabling end-to-end pose optimization")

## 4. GS-SLAM Architecture

GS-SLAM improves upon SplaTAM with several optimizations, especially coarse-to-fine tracking.

In [ ]:
# GS-SLAM Architecture
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('GS-SLAM: Coarse-to-Fine Tracking Strategy', fontsize=16, fontweight='bold')

# Pyramid structure
levels = [
    {'name': 'Coarse Level', 'res': '1/8 Resolution', 'iters': 10, 'lr': 0.02, 'y': 9},
    {'name': 'Medium Level', 'res': '1/4 Resolution', 'iters': 15, 'lr': 0.01, 'y': 6},
    {'name': 'Fine Level', 'res': 'Full Resolution', 'iters': 20, 'lr': 0.005, 'y': 3},
]

colors = ['#FFCDD2', '#FFE0B2', '#C8E6C9']

for i, (level, color) in enumerate(zip(levels, colors)):
    box = FancyBboxPatch((2, level['y']-0.8), 10, 2, boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(box)
    
    ax.text(7, level['y']+0.8, level['name'], ha='center', fontsize=12, fontweight='bold')
    ax.text(4, level['y']+0.2, f"Resolution: {level['res']}", ha='left', fontsize=10)
    ax.text(4, level['y']-0.1, f"Iterations: {level['iters']}", ha='left', fontsize=10)
    ax.text(4, level['y']-0.4, f"Learning Rate: {level['lr']}", ha='left', fontsize=10)
    
    ax.text(9, level['y']+0.2, 'Render -> Compare -> Optimize', ha='left', fontsize=9, style='italic')

# Connect arrows
ax.annotate('', xy=(7, 6.2), xytext=(7, 8.2),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=3))
ax.text(8, 7.2, 'Refine', fontsize=10, fontweight='bold', color='#1565C0')

ax.annotate('', xy=(7, 3.2), xytext=(7, 5.2),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=3))
ax.text(8, 4.2, 'Refine', fontsize=10, fontweight='bold', color='#1565C0')

ax.text(7, 10.5, 'Initial Pose Estimate', ha='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(7, 9.2), xytext=(7, 10.2),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

output_box = FancyBboxPatch((5, 0.5), 4, 1, boxstyle="round,pad=0.1",
                            facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(output_box)
ax.text(7, 1.0, 'Final Optimized Pose', ha='center', fontsize=11, fontweight='bold')

adv_box = FancyBboxPatch((0.5, 0.3), 13, 1.8, boxstyle="round,pad=0.1",
                         facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(adv_box)
ax.text(7, 1.8, 'GS-SLAM Advantages', ha='center', fontsize=12, fontweight='bold', color='#F57F17')
ax.text(7, 1.3, 'Coarse-to-fine: Fast convergence + Accurate optimization | Adaptive Gaussian expansion: Balance accuracy and efficiency',
        ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('gs_slam_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGS-SLAM Core Improvements:")
print("  1. Coarse-to-fine tracking: Multi-resolution progressive optimization for faster convergence")
print("  2. Adaptive Gaussian expansion: Dynamically add Gaussians based on uncertainty")
print("  3. Optimized keyframe selection: More robust keyframe strategy")
print("  4. Overall ~2x faster than SplaTAM")

## 5. MonoGS: Monocular 3DGS SLAM

MonoGS addresses the limitation of SplaTAM and GS-SLAM: no depth sensor required!

In [ ]:
# MonoGS Architecture
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('MonoGS: Monocular 3DGS SLAM', fontsize=16, fontweight='bold')

# Input
input_box = FancyBboxPatch((5.5, 10), 3, 1, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(7, 10.5, 'RGB Only', ha='center', fontsize=12, fontweight='bold')

# Depth Estimation Module
depth_box = FancyBboxPatch((1, 7.5), 5, 2, boxstyle="round,pad=0.15",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)
ax.add_patch(depth_box)
ax.text(3.5, 9.0, 'Depth Estimation', ha='center', fontsize=12, fontweight='bold', color='#E65100')
ax.text(3.5, 8.5, '• Monocular Depth Network', ha='center', fontsize=9)
ax.text(3.5, 8.2, '• Scale Ambiguity Handling', ha='center', fontsize=9)
ax.text(3.5, 7.9, '• Multi-view Consistency', ha='center', fontsize=9)

# Joint Optimization Module
joint_box = FancyBboxPatch((8, 7.5), 5, 2, boxstyle="round,pad=0.15",
                           facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=3)
ax.add_patch(joint_box)
ax.text(10.5, 9.0, 'Joint Optimization', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ax.text(10.5, 8.5, '• Camera Pose', ha='center', fontsize=9)
ax.text(10.5, 8.2, '• Gaussian Parameters', ha='center', fontsize=9)
ax.text(10.5, 7.9, '• Depth Scale', ha='center', fontsize=9)

# Loop Closure
loop_box = FancyBboxPatch((5, 5), 4, 1.5, boxstyle="round,pad=0.1",
                          facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=2)
ax.add_patch(loop_box)
ax.text(7, 5.9, 'Loop Closure', ha='center', fontsize=11, fontweight='bold', color='#6A1B9A')
ax.text(7, 5.4, '• Place Recognition\n• Global Bundle Adjustment', ha='center', fontsize=8)

# Gaussian Map
map_box = FancyBboxPatch((5, 2.5), 4, 2, boxstyle="round,pad=0.1",
                         facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2)
ax.add_patch(map_box)
ax.text(7, 4.0, 'Gaussian Map', ha='center', fontsize=11, fontweight='bold')
ax.text(7, 3.5, '(Scale-Adjusted)', ha='center', fontsize=9, style='italic')

# Arrows
ax.annotate('', xy=(3.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(10.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(7, 6.5), xytext=(3.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
ax.annotate('', xy=(7, 6.5), xytext=(10.5, 7.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.annotate('', xy=(7, 4.5), xytext=(7, 5),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=2))

challenge_box = FancyBboxPatch((0.5, 0.3), 13, 1.8, boxstyle="round,pad=0.1",
                               facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(challenge_box)
ax.text(7, 1.8, 'MonoGS Challenges and Solutions', ha='center', fontsize=12, fontweight='bold')
ax.text(7, 1.3, 'Challenge: Monocular depth scale ambiguity | Solution: Multi-view geometric constraints + Joint depth scale optimization',
        ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('monogs_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMonoGS Core Features:")
print("  1. Pure RGB input: No depth sensor required")
print("  2. Monocular depth estimation: Network prediction + Multi-view constraints")
print("  3. Scale handling: Joint optimization of depth scale parameters")
print("  4. Loop closure: Explicit loop closure to correct drift")
print("  5. Application: Scenarios where depth data is unavailable")

## 6. Photo-SLAM: Hybrid Approach

Photo-SLAM combines the advantages of traditional SLAM feature tracking with 3DGS rendering.

In [ ]:
# Photo-SLAM Architecture
fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Photo-SLAM: Hybrid Feature-based + 3DGS Approach', fontsize=16, fontweight='bold')

# Input
input_box = FancyBboxPatch((5.5, 10), 3, 1, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(input_box)
ax.text(7, 10.5, 'RGB Input', ha='center', fontsize=12, fontweight='bold')

# ORB Feature Tracking
orb_box = FancyBboxPatch((1, 7), 5, 2.5, boxstyle="round,pad=0.15",
                         facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=3)
ax.add_patch(orb_box)
ax.text(3.5, 9, 'ORB Feature Tracking', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ax.text(3.5, 8.5, '• Fast Feature Extraction', ha='center', fontsize=9)
ax.text(3.5, 8.2, '• Feature Matching', ha='center', fontsize=9)
ax.text(3.5, 7.9, '• PnP Pose Estimation', ha='center', fontsize=9)
ax.text(3.5, 7.6, '• Local Bundle Adjustment', ha='center', fontsize=9)
ax.text(3.5, 7.3, '(Traditional SLAM)', ha='center', fontsize=8, style='italic', color='gray')

# Neural Rendering
render_box = FancyBboxPatch((8, 7), 5, 2.5, boxstyle="round,pad=0.15",
                            facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)
ax.add_patch(render_box)
ax.text(10.5, 9, 'Neural Rendering', ha='center', fontsize=12, fontweight='bold', color='#E65100')
ax.text(10.5, 8.5, '• Keyframe Gaussian Map', ha='center', fontsize=9)
ax.text(10.5, 8.2, '• Differentiable Rendering', ha='center', fontsize=9)
ax.text(10.5, 7.9, '• Photometric Refinement', ha='center', fontsize=9)
ax.text(10.5, 7.6, '• Superpoint-based', ha='center', fontsize=9)
ax.text(10.5, 7.3, '(3DGS Rendering)', ha='center', fontsize=8, style='italic', color='gray')

# Fusion Module
fuse_box = FancyBboxPatch((5, 4.5), 4, 2, boxstyle="round,pad=0.1",
                          facecolor='#F3E5F5', edgecolor='#6A1B9A', linewidth=2)
ax.add_patch(fuse_box)
ax.text(7, 6, 'Fusion Module', ha='center', fontsize=11, fontweight='bold', color='#6A1B9A')
ax.text(7, 5.5, '• Feature Points -> Sparse Map', ha='center', fontsize=9)
ax.text(7, 5.2, '• Gaussians -> Dense Rendering', ha='center', fontsize=9)
ax.text(7, 4.9, '• Joint Optimization', ha='center', fontsize=9)

# Output
output_box = FancyBboxPatch((5, 2), 4, 1.5, boxstyle="round,pad=0.1",
                            facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2)
ax.add_patch(output_box)
ax.text(7, 3.0, 'Hybrid Map Output', ha='center', fontsize=11, fontweight='bold')
ax.text(7, 2.6, 'Sparse Points + Dense Gaussians', ha='center', fontsize=9)

# Arrows
ax.annotate('', xy=(3.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(10.5, 9.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.annotate('', xy=(6, 6.5), xytext=(3.5, 7),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.text(4.5, 6.8, 'Pose', fontsize=9, color='#2E7D32')
ax.annotate('', xy=(8, 6.5), xytext=(10.5, 7),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
ax.text(9.5, 6.8, 'Render', fontsize=9, color='#E65100')
ax.annotate('', xy=(7, 3.5), xytext=(7, 4.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

adv_box = FancyBboxPatch((0.5, 0.3), 13, 1.5, boxstyle="round,pad=0.1",
                         facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(adv_box)
ax.text(7, 1.5, 'Photo-SLAM Advantages', ha='center', fontsize=12, fontweight='bold')
ax.text(7, 1.0, 'Best of both: ORB robust tracking + 3DGS photorealistic rendering', ha='center', fontsize=10)
ax.text(7, 0.6, 'Superpoint provides stable features, rendering module provides high-quality visualization', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('photoslam_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPhoto-SLAM Core Features:")
print("  1. Hybrid architecture: ORB feature tracking + Gaussian neural rendering")
print("  2. Dual map representation: Sparse feature points + Dense Gaussians")
print("  3. Superpoint features: More robust than ORB")
print("  4. Real-time performance: Traditional SLAM speed + Neural rendering quality")
print("  5. Application: Applications requiring high-precision tracking and quality rendering")

## 7. Comprehensive Comparison

Let's compare the four methods across multiple dimensions.

In [ ]:
# Radar chart comparison
from math import pi

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

categories = ['Tracking\nAccuracy', 'Mapping\nQuality', 'Rendering\nQuality', 'Real-time\nPerf', 'Scalability', 'Ease of\nUse']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Scores (1-10)
scores = {
    'SplaTAM': [7, 8, 9, 7, 6, 9],
    'GS-SLAM': [8, 8, 9, 8, 7, 7],
    'MonoGS': [7, 7, 8, 7, 7, 6],
    'Photo-SLAM': [9, 8, 9, 9, 7, 7],
}

colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']

for idx, (method, color) in enumerate(zip(scores.keys(), colors)):
    ax = axes[idx // 2, idx % 2]
    
    values = scores[method]
    values += values[:1]
    
    ax = plt.subplot(2, 2, idx+1, projection='polar')
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=method)
    ax.fill(angles, values, alpha=0.25, color=color)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=9)
    ax.set_ylim(0, 10)
    ax.set_title(f'{method}', fontsize=14, fontweight='bold', color=color, pad=20)
    ax.grid(True)

plt.tight_layout()
plt.savefig('radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRadar Chart Explanation (1-10 scale):")
print("  • Tracking Accuracy: Pose estimation precision")
print("  • Mapping Quality: Map geometric quality")
print("  • Rendering Quality: Rendering photorealism")
print("  • Real-time Performance: Real-time capability")
print("  • Scalability: Large-scale scene extensibility")
print("  • Ease of Use: Ease of use and code availability")

## 8. Performance Benchmarks

Let's look at how these methods perform on standard datasets.

In [ ]:
# Performance benchmarks
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ATE RMSE on TUM fr1/desk (cm)
ax = axes[0, 0]
methods = ['ORB-SLAM2', 'SplaTAM', 'GS-SLAM', 'MonoGS', 'Photo-SLAM']
ate_rmse = [1.60, 3.35, 2.80, 4.20, 2.10]
colors_bar = ['gray', '#1565C0', '#2E7D32', '#E65100', '#6A1B9A']

bars = ax.bar(methods, ate_rmse, color=colors_bar, alpha=0.8, edgecolor='black')
ax.set_ylabel('ATE RMSE (cm)', fontsize=12, fontweight='bold')
ax.set_title('TUM fr1/desk: Tracking Accuracy\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 5)

for bar, val in zip(bars, ate_rmse):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Rendering PSNR on Replica (dB)
ax = axes[0, 1]
scenes = ['Room0', 'Room1', 'Room2', 'Office0', 'Office1']
splatam_psnr = [32.5, 33.1, 31.8, 30.2, 29.8]
gsslam_psnr = [33.2, 33.8, 32.5, 31.0, 30.5]
monogs_psnr = [28.5, 29.2, 28.0, 27.5, 27.0]
photoslam_psnr = [31.8, 32.5, 31.2, 29.8, 29.2]

x = np.arange(len(scenes))
width = 0.2

ax.bar(x - 1.5*width, splatam_psnr, width, label='SplaTAM', color='#1565C0', alpha=0.8)
ax.bar(x - 0.5*width, gsslam_psnr, width, label='GS-SLAM', color='#2E7D32', alpha=0.8)
ax.bar(x + 0.5*width, monogs_psnr, width, label='MonoGS', color='#E65100', alpha=0.8)
ax.bar(x + 1.5*width, photoslam_psnr, width, label='Photo-SLAM', color='#6A1B9A', alpha=0.8)

ax.set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
ax.set_title('Replica Dataset: Rendering Quality\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenes)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Tracking FPS comparison
ax = axes[1, 0]
tracking_fps = [10, 20, 15, 30]
method_names = ['SplaTAM', 'GS-SLAM', 'MonoGS', 'Photo-SLAM']
colors_fps = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A']

bars = ax.bar(method_names, tracking_fps, color=colors_fps, alpha=0.8, edgecolor='black')
ax.set_ylabel('Tracking FPS', fontsize=12, fontweight='bold')
ax.set_title('Tracking Speed Comparison\n(Higher is Better)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 35)

ax.axhline(30, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Real-time (30 FPS)')
ax.legend()

for bar, val in zip(bars, tracking_fps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{val}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Memory usage comparison (GB)
ax = axes[1, 1]
memory_gb = [8, 10, 12, 9]

bars = ax.bar(method_names, memory_gb, color=colors_fps, alpha=0.8, edgecolor='black')
ax.set_ylabel('GPU Memory (GB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Usage Comparison\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 15)

ax.axhline(24, color='green', linestyle='--', linewidth=2, alpha=0.5, label='RTX 3090 (24GB)')
ax.legend()

for bar, val in zip(bars, memory_gb):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, f'{val}GB',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('performance_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPerformance Benchmark Summary:")
print("  Tracking Accuracy (ATE RMSE):")
print("    • Photo-SLAM best (2.10cm) - combines traditional feature tracking precision")
print("    • SplaTAM (3.35cm) and GS-SLAM (2.80cm) slightly lower than ORB-SLAM2")
print("    • MonoGS lower precision (4.20cm) due to monocular scale ambiguity")
print("\n  Rendering Quality (PSNR):")
print("    • GS-SLAM best (~32dB) - thanks to adaptive optimization strategy")
print("    • SplaTAM and Photo-SLAM similar (~31-32dB)")
print("    • MonoGS slightly lower (~28dB) due to depth estimation error")
print("\n  Real-time Performance:")
print("    • Photo-SLAM fastest (30 FPS) - efficient traditional feature tracking")
print("    • GS-SLAM second (20 FPS) - coarse-to-fine strategy acceleration")
print("    • MonoGS and SplaTAM at 10-15 FPS")

## 9. Method Selection Guide

How to choose the right method for different scenarios?

In [ ]:
# Decision tree
fig, ax = plt.subplots(figsize=(16, 14))
ax.set_xlim(0, 16)
ax.set_ylim(0, 15)
ax.axis('off')
ax.set_title('When to Use Which 3DGS+SLAM Method', fontsize=18, fontweight='bold')

# Decision nodes
decisions = [
    {'q': 'Do you have RGB-D camera?', 'y': 13, 'x': 8},
    {'q': 'Need real-time performance?', 'y': 10, 'x': 4},
    {'q': 'Need loop closure?', 'y': 10, 'x': 12},
]

for d in decisions:
    box = FancyBboxPatch((d['x']-2, d['y']-0.3), 4, 0.6, boxstyle="round,pad=0.1",
                         facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
    ax.add_patch(box)
    ax.text(d['x'], d['y'], d['q'], ha='center', va='center', fontsize=10, fontweight='bold')

# Connection lines
ax.plot([8, 4], [12.7, 10.3], 'k-', linewidth=2)
ax.plot([8, 12], [12.7, 10.3], 'k-', linewidth=2)
ax.text(5.5, 11.5, 'Yes', fontsize=9, fontweight='bold', color='green')
ax.text(10, 11.5, 'No', fontsize=9, fontweight='bold', color='red')

# Method recommendations
recommendations = [
    {'cond': 'RGB-D + Fast', 'method': 'GS-SLAM', 'color': '#2E7D32',
     'x': 2, 'y': 7, 'reason': 'Coarse-to-fine tracking\nfor speed'},
    {'cond': 'RGB-D + Accurate', 'method': 'SplaTAM', 'color': '#1565C0',
     'x': 6, 'y': 7, 'reason': 'Classic approach\nWell documented'},
    {'cond': 'RGB + Loop Closure', 'method': 'MonoGS', 'color': '#E65100',
     'x': 10, 'y': 7, 'reason': 'Explicit loop closure\nScalable'},
    {'cond': 'RGB + Robust', 'method': 'Photo-SLAM', 'color': '#6A1B9A',
     'x': 14, 'y': 7, 'reason': 'Feature-based tracking\nBest accuracy'},
]

for rec in recommendations:
    cond_box = FancyBboxPatch((rec['x']-1.5, rec['y']+1.8), 3, 0.6, boxstyle="round,pad=0.05",
                              facecolor='#FFF9C4', edgecolor='gray', linewidth=1)
    ax.add_patch(cond_box)
    ax.text(rec['x'], rec['y']+2.1, rec['cond'], ha='center', fontsize=9, style='italic')
    
    method_box = FancyBboxPatch((rec['x']-1.5, rec['y']-0.5), 3, 2, boxstyle="round,pad=0.1",
                                facecolor='white', edgecolor=rec['color'], linewidth=3)
    ax.add_patch(method_box)
    ax.text(rec['x'], rec['y']+1.0, rec['method'], ha='center', fontsize=13, fontweight='bold',
            color=rec['color'])
    ax.text(rec['x'], rec['y']+0.3, rec['reason'], ha='center', fontsize=8, linespacing=1.3)

# Connect to methods
ax.plot([4, 2], [9.7, 7.5], 'k-', linewidth=1.5)
ax.plot([4, 6], [9.7, 7.5], 'k-', linewidth=1.5)
ax.plot([12, 10], [9.7, 7.5], 'k-', linewidth=1.5)
ax.plot([12, 14], [9.7, 7.5], 'k-', linewidth=1.5)

# Application scenarios
scenarios_box = FancyBboxPatch((0.5, 0.5), 15, 4.5, boxstyle="round,pad=0.1",
                               facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(scenarios_box)
ax.text(8, 4.6, 'Real-World Application Scenarios', ha='center', fontsize=13, fontweight='bold')

scenarios = [
    ('Indoor Mapping', 'SplaTAM/GS-SLAM', 'Static environment, RGB-D available'),
    ('Drone Aerial', 'MonoGS', 'No depth sensor, large-scale'),
    ('Robot Navigation', 'Photo-SLAM', 'Robust tracking critical'),
    ('AR/VR', 'GS-SLAM', 'Real-time requirement'),
    ('Cultural Heritage', 'SplaTAM', 'High quality reconstruction'),
]

for i, (scene, method, desc) in enumerate(scenarios):
    y = 3.8 - i * 0.65
    ax.text(1.5, y, scene, ha='left', fontsize=10, fontweight='bold')
    ax.text(5.5, y, '->', ha='center', fontsize=12)
    ax.text(6.5, y, method, ha='left', fontsize=10, color='#1565C0')
    ax.text(10.5, y, f'({desc})', ha='left', fontsize=8, color='gray', style='italic')

plt.tight_layout()
plt.savefig('method_selection_guide.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMethod Selection Guide:")
print("  Have RGB-D camera and need speed:   -> GS-SLAM")
print("  Have RGB-D camera and need stability: -> SplaTAM")
print("  RGB-only camera and large scene:    -> MonoGS")
print("  RGB-only camera and need accuracy:  -> Photo-SLAM")

## 10. Summary

Let's summarize the key differences and takeaways.

In [ ]:
# Final summary
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_xlim(0, 16)
ax.set_ylim(0, 14)
ax.axis('off')

ax.text(8, 13.2, '3DGS+SLAM Methods Comparison Summary', ha='center', fontsize=18, fontweight='bold')

content_box = FancyBboxPatch((0.5, 1), 15, 11.5, boxstyle="round,pad=0.1",
                             facecolor='white', edgecolor='#1565C0', linewidth=3)
ax.add_patch(content_box)

key_points = [
    ('1. Method Evolution', '2023-2024 saw rapid development of 3DGS+SLAM methods'),
    ('2. Input Types', 'RGB-D (SplaTAM, GS-SLAM) vs RGB-only (MonoGS, Photo-SLAM)'),
    ('3. Tracking Paradigm', 'Differentiable rendering replaces traditional geometric error'),
    ('4. Performance Trade-offs', 'Speed vs accuracy: GS-SLAM fast, Photo-SLAM accurate'),
    ('5. Rendering Quality', 'All methods achieve photo-realistic rendering (~30dB PSNR)'),
    ('6. Practical Choice', 'Hardware availability and application requirements drive selection'),
]

y_start = 11.5
for title, desc in key_points:
    ax.text(1.5, y_start, f'• {title}:', ha='left', fontsize=12, fontweight='bold')
    ax.text(1.8, y_start-0.4, desc, ha='left', fontsize=10)
    y_start -= 1.3

ref_box = FancyBboxPatch((1, 2.5), 14, 3, boxstyle="round,pad=0.1",
                         facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(ref_box)
ax.text(8, 5.2, 'Quick Reference Guide', ha='center', fontsize=13, fontweight='bold')

ref_items = [
    'SplaTAM: Best for learning and research (well documented, open source)',
    'GS-SLAM: Best for speed with RGB-D (coarse-to-fine optimization)',
    'MonoGS: Best for RGB-only large scenes (monocular depth estimation)',
    'Photo-SLAM: Best for accuracy and robustness (hybrid feature-based)',
]

for i, item in enumerate(ref_items):
    ax.text(1.5, 4.6-i*0.5, f'-> {item}', ha='left', fontsize=9)

next_box = FancyBboxPatch((0.5, 0.1), 15, 0.8, boxstyle="round,pad=0.1",
                          facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2)
ax.add_patch(next_box)
ax.text(8, 0.7, 'Next Steps: Run experiments on your data, compare methods, and contribute to the community!',
        ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('final_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*80)
print("3DGS+SLAM Methods Comparison Summary".center(80))
print("="*80)
print("\nKey Points:")
print("  1. SplaTAM: First complete 3DGS SLAM, suitable for learning and research")
print("  2. GS-SLAM: Fastest, suitable for real-time applications")
print("  3. MonoGS: RGB-only, suitable for scenarios without depth sensor")
print("  4. Photo-SLAM: Highest accuracy, suitable for robustness-critical applications")
print("\nSelection Advice:")
print("  • Beginners -> SplaTAM (open source, well documented)")
print("  • Real-time applications -> GS-SLAM or Photo-SLAM")
print("  • No depth sensor -> MonoGS")
print("  • High accuracy requirements -> Photo-SLAM")
print("\n" + "="*80)

---

## Exercises

1. **Method Comparison**: Based on your application scenario, choose the most suitable method and explain why
2. **Performance Analysis**: If you have Intel RealSense D435i, which method would you choose? Why?
3. **Extension Thinking**: If you need to handle dynamic scenes, what improvements do these methods need?

## References

1. **SplaTAM**: "Splat, Track & Map 3D Gaussians for Dense RGB-D SLAM" - CVPR 2024
2. **GS-SLAM**: "GS-SLAM: Dense Visual SLAM with 3D Gaussian Splatting" - CVPR 2024
3. **MonoGS**: "Gaussian Splatting SLAM" - CVPR 2024
4. **Photo-SLAM**: "Photo-SLAM: Real-time Simultaneous Localization and Photorealistic Mapping" - CVPR 2024

---

*Notebook created for Phase 2: 3DGS+SLAM course*

**Previous**: [04_gaussian_update.ipynb](./04_gaussian_update.ipynb) | **Next**: TBD